# Modelo de Predicción de Churn — Opción B (Temporal)

## Caso Práctico - Empresa de Telecomunicaciones
## Prácticas Aplicadas 2026

---

## ¿Por qué un modelo temporal?

En el modelo binario (Opción A) teníamos una fila por cliente con toda su historia resumida. Ese enfoque tiene un problema metodológico importante: **mezcla información de distintos momentos en el tiempo**.

El caso más claro es `n_meses_facturados`: sabemos cuántos meses estuvo facturando el cliente solo porque ya sabemos si se fue o no. Es información del futuro colándose en el modelo — lo que se llama **leakage**. Eso infla artificialmente el AUC.

En el modelo temporal resolvemos esto trabajando con **una fila por cliente y mes**:
- La etiqueta es `churn(t)`: ¿se fue este cliente en el mes t?
- Las features son siempre información de meses anteriores: `impago(t-1)`, `stress(t-1)`, `interacciones en los últimos 3 meses`...

Así el modelo aprende a predecir el churn usando solo lo que sabríamos en ese momento, sin ver el futuro. Es la forma correcta de hacerlo en producción.

**Comparativa de enfoques:**

| | Modelo Binario (A) | Modelo Temporal (B) |
|---|---|---|
| Estructura | 1 fila por cliente | 1 fila por cliente-mes |
| Leakage | Sí (variables acumuladas) | No (lags correctos) |
| AUC esperado | Alto (inflado) | Más real |
| Uso en producción | No recomendable | Sí |
| Predicción | Estática | Mensual |


## 1. Librerías


In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.load import load_all
from src.clean import clean_all

RANDOM_STATE = 42
print('Librerías cargadas')

## 2. Carga de datos


In [ ]:
data  = load_all()
clean = clean_all(data, save=False)

clientes  = clean['clientes']
churn     = clean['churn']
factura   = clean['facturacion']
soporte   = clean['soporte']
calidad   = clean['calidad']

print('\nDatos cargados y limpios')

---
## 3. Construcción del panel temporal

El panel tiene una fila por cliente-mes. El proceso es:
1. Tomamos el panel de churn como base (ya tiene la estructura correcta)
2. Agregamos la facturación a nivel mes y aplicamos lag de 1 mes
3. Agregamos el soporte a nivel mes y aplicamos lag de 1 mes
4. Añadimos rolling windows (media de los últimos 3 meses)
5. Unimos el perfil estático del cliente y la calidad de red por zona

**Principio fundamental**: para predecir `churn(t)`, usamos solo features de `t-1` o anteriores.


In [ ]:
# ── Paso 1: Facturación agregada a nivel cliente-mes ──────────────────────
# La facturación puede tener varios registros por mes (distintas fechas de corte)
# Los agregamos a nivel mes antes de hacer el lag

factura['mes'] = factura['fecha'].dt.to_period('M').dt.to_timestamp()

factura_mes = factura.groupby(['cliente_id', 'mes']).agg(
    importe_mes       = ('importe_total',        'sum'),
    impago_mes        = ('impago_flag',           'max'),   # 1 si hubo algún impago ese mes
    dias_retraso_mes  = ('dias_retraso_pago',     'max'),
    stress_mes        = ('stress_calidad_lag',    'mean'),
    consumo_extra_mes = ('consumo_extra',         'sum'),
    variacion_consumo = ('variacion_consumo_pct', 'mean'),
    descuento_mes     = ('descuento_aplicado',    'sum'),
).reset_index()

print(f'Facturación mensual: {len(factura_mes):,} registros cliente-mes')
print(f'Clientes únicos: {factura_mes["cliente_id"].nunique():,}')

In [ ]:
# ── Paso 2: Soporte agregado a nivel cliente-mes ──────────────────────────
soporte['mes_soporte'] = soporte['mes'].dt.to_period('M').dt.to_timestamp()

soporte_mes = soporte.groupby(['cliente_id', 'mes_soporte']).agg(
    n_interacciones_mes = ('interaccion_id',    'count'),
    resolucion_mes      = ('resuelto',          'mean'),
    satisfaccion_mes    = ('satisfaccion_post', 'mean'),
    n_baja_mes          = ('motivo', lambda x: (x == 'Baja / portabilidad').sum()),
).reset_index().rename(columns={'mes_soporte': 'mes'})

print(f'Soporte mensual: {len(soporte_mes):,} registros cliente-mes')

In [ ]:
# ── Paso 3: Panel base desde churn ────────────────────────────────────────
panel = churn.copy()
panel['mes'] = panel['fecha'].dt.to_period('M').dt.to_timestamp()

# ── Paso 4: Aplicar LAG 1 en facturación ─────────────────────────────────
# Para predecir churn en el mes t, usamos la facturación del mes t-1
# Esto lo hacemos desplazando la fecha: el mes t-1 se une al mes t del panel

factura_lag = factura_mes.copy()
factura_lag['mes'] = factura_lag['mes'] + pd.DateOffset(months=1)
factura_lag = factura_lag.rename(columns={
    col: f'{col}_lag1' for col in factura_lag.columns
    if col not in ['cliente_id', 'mes']
})

panel = panel.merge(factura_lag, on=['cliente_id', 'mes'], how='left')

# ── Paso 5: Rolling window — media de los últimos 3 meses ─────────────────
# Para capturar tendencias recientes, calculamos la media móvil de 3 meses
# También con lag para no usar el mes actual

factura_sorted = factura_mes.sort_values(['cliente_id', 'mes'])
for col in ['impago_mes', 'stress_mes', 'variacion_consumo']:
    factura_sorted[f'{col}_roll3'] = (
        factura_sorted.groupby('cliente_id')[col]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

roll_cols = ['cliente_id', 'mes', 'impago_mes_roll3', 'stress_mes_roll3', 'variacion_consumo_roll3']
factura_roll = factura_sorted[roll_cols].copy()
factura_roll['mes'] = factura_roll['mes'] + pd.DateOffset(months=1)
panel = panel.merge(factura_roll, on=['cliente_id', 'mes'], how='left')

# ── Paso 6: Aplicar LAG 1 en soporte ─────────────────────────────────────
soporte_lag = soporte_mes.copy()
soporte_lag['mes'] = soporte_lag['mes'] + pd.DateOffset(months=1)
soporte_lag = soporte_lag.rename(columns={
    col: f'{col}_lag1' for col in soporte_lag.columns
    if col not in ['cliente_id', 'mes']
})
panel = panel.merge(soporte_lag, on=['cliente_id', 'mes'], how='left')

# Clientes sin soporte ese mes → 0
for col in ['n_interacciones_mes_lag1', 'n_baja_mes_lag1']:
    panel[col] = panel[col].fillna(0)

# ── Paso 7: Calidad de red por zona-mes ───────────────────────────────────
calidad['mes'] = calidad['fecha'].dt.to_period('M').dt.to_timestamp()
calidad_lag = calidad[['zona_id', 'mes', 'indice_calidad_global', 'tasa_cortes_pct',
                         'cobertura_5g_pct']].copy()
calidad_lag['mes'] = calidad_lag['mes'] + pd.DateOffset(months=1)
calidad_lag = calidad_lag.rename(columns={
    'indice_calidad_global': 'calidad_global_lag1',
    'tasa_cortes_pct':       'tasa_cortes_lag1',
    'cobertura_5g_pct':      'cobertura_5g_lag1',
})

zona_cliente = clientes[['cliente_id', 'zona_id']]
panel = panel.merge(zona_cliente, on='cliente_id', how='left')
panel = panel.merge(calidad_lag, on=['zona_id', 'mes'], how='left')

# ── Paso 8: Perfil estático del cliente ───────────────────────────────────
cols_perfil = ['cliente_id', 'edad', 'sexo', 'estado_civil', 'num_lineas',
               'tipo_plan', 'tipo_zona', 'region', 'ingreso_estimado',
               'antiguedad_meses', 'descuento_activo', 'tipo_dispositivo']
cols_perfil_disp = [c for c in cols_perfil if c in clientes.columns]
panel = panel.merge(clientes[cols_perfil_disp], on='cliente_id', how='left')

# Mapeo ordinal de tipo_plan
MAPA_PLAN = {'Básico': 1, 'Estándar': 2, 'Premium': 3}
panel['tipo_plan_num'] = panel['tipo_plan'].map(MAPA_PLAN)

print(f'Panel temporal: {panel.shape[0]:,} filas x {panel.shape[1]} columnas')
print(f'Clientes únicos: {panel["cliente_id"].nunique():,}')
print(f'Tasa de churn en panel: {panel["churn"].mean()*100:.2f}%')
print(f'Filas con churn=1: {panel["churn"].sum():,}')

---
## 4. Selección de variables

Usamos las mismas tres categorías que en el modelo binario: numéricas continuas, nominales y ordinales. La diferencia es que ahora todas las variables son con lag — información del pasado.


In [ ]:
# Variables numéricas — todas con lag o estáticas del perfil
FEATURES_NUM = [
    # Perfil estático
    'edad', 'num_lineas', 'ingreso_estimado', 'antiguedad_meses', 'descuento_activo',
    # Facturación lag 1
    'importe_mes_lag1', 'impago_mes_lag1', 'dias_retraso_mes_lag1',
    'stress_mes_lag1', 'consumo_extra_mes_lag1', 'variacion_consumo_lag1',
    # Rolling 3 meses
    'impago_mes_roll3', 'stress_mes_roll3', 'variacion_consumo_roll3',
    # Soporte lag 1
    'n_interacciones_mes_lag1', 'resolucion_mes_lag1',
    'satisfaccion_mes_lag1', 'n_baja_mes_lag1',
    # Calidad de red lag 1
    'calidad_global_lag1', 'tasa_cortes_lag1', 'cobertura_5g_lag1',
]

# Variables nominales — sin orden
FEATURES_CAT_NOMINAL = [
    'region', 'tipo_zona', 'sexo', 'estado_civil', 'tipo_dispositivo',
]

# Variable ordinal — tipo de plan con orden
FEATURES_ORDINAL = ['tipo_plan_num']

TARGET = 'churn'

all_features = FEATURES_NUM + FEATURES_CAT_NOMINAL + FEATURES_ORDINAL
missing = [f for f in all_features if f not in panel.columns]

if missing:
    print(f'⚠️ Columnas no encontradas: {missing}')
else:
    print('✅ Todas las variables disponibles')
    print(f'   Numéricas con lag:  {len(FEATURES_NUM)}')
    print(f'   Nominales:          {len(FEATURES_CAT_NOMINAL)}')
    print(f'   Ordinales:          {len(FEATURES_ORDINAL)}')

# Resumen de nulos
nulos = panel[all_features].isnull().sum()
print(f'\nColumnas con nulos (primeros meses sin lag disponible):')
print(nulos[nulos > 0])

---
## 5. División train / test

En un panel temporal no podemos hacer un split aleatorio porque mezclaríamos meses del futuro en el entrenamiento. Hay dos opciones:

- **Split temporal**: entrenar con meses anteriores a una fecha de corte, testear con meses posteriores
- **Split por cliente**: entrenar con un subconjunto de clientes, testear con otro

Usamos el **split por cliente** porque es más sencillo de implementar y evita el problema de que el mismo cliente aparezca en train y test. Los clientes de test nunca fueron vistos en el entrenamiento.


In [ ]:
# Split por cliente: 80% de clientes para train, 20% para test
clientes_unicos = panel['cliente_id'].unique()
np.random.seed(RANDOM_STATE)
clientes_test = np.random.choice(
    clientes_unicos,
    size=int(len(clientes_unicos) * 0.2),
    replace=False
)
clientes_train = np.setdiff1d(clientes_unicos, clientes_test)

train = panel[panel['cliente_id'].isin(clientes_train)].copy()
test  = panel[panel['cliente_id'].isin(clientes_test)].copy()

# Eliminamos las filas del primer mes (no tienen lag disponible)
train = train.dropna(subset=['importe_mes_lag1', 'stress_mes_lag1'])
test  = test.dropna(subset=['importe_mes_lag1', 'stress_mes_lag1'])

X_train = train[all_features]
y_train = train[TARGET]
X_test  = test[all_features]
y_test  = test[TARGET]

print(f'Train: {len(X_train):,} filas | {y_train.mean()*100:.2f}% churn')
print(f'Test:  {len(X_test):,} filas  | {y_test.mean()*100:.2f}% churn')
print(f'Clientes en train: {len(clientes_train):,}')
print(f'Clientes en test:  {len(clientes_test):,}')

## 6. Preprocesado

Mismo criterio que en el modelo binario: tres pipelines según la naturaleza de cada variable.


In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='desconocido')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])

ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, FEATURES_NUM),
    ('nom', nom_pipeline, FEATURES_CAT_NOMINAL),
    ('ord', ord_pipeline, FEATURES_ORDINAL),
])

print('Preprocesador definido')
print('  num → mediana + StandardScaler    (numéricas con lag)')
print('  nom → desconocido + OneHotEncoder  (nominales sin orden)')
print('  ord → mediana + StandardScaler    (ordinales con orden)')

---
## 7. Modelo 1 — Regresión Logística (Baseline)


In [ ]:
pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, C=1.0))
])

pipe_lr.fit(X_train, y_train)
y_pred_proba_lr = pipe_lr.predict_proba(X_test)[:, 1]
auc_lr = roc_auc_score(y_test, y_pred_proba_lr)

print(f'Regresión Logística — AUC en test: {auc_lr:.3f}')

# Coeficientes
nom_feature_names = (pipe_lr.named_steps['preprocessor']
                     .named_transformers_['nom']
                     .named_steps['encoder']
                     .get_feature_names_out(FEATURES_CAT_NOMINAL))
feature_names = FEATURES_NUM + list(nom_feature_names) + FEATURES_ORDINAL

coefs = pd.Series(
    pipe_lr.named_steps['model'].coef_[0],
    index=feature_names
).sort_values(key=abs, ascending=False)

top20 = coefs.head(20).sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#E85C4C' if v > 0 else '#4C9BE8' for v in top20.values]
ax.barh(top20.index, top20.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Regresión Logística Temporal — Top 20 variables\n'
             '(rojo = aumenta churn | azul = lo reduce)',
             fontweight='bold')
ax.set_xlabel('Coeficiente (escala estandarizada)')
plt.tight_layout()
plt.show()

### Resultado — Regresión Logística Temporal

**AUC en test: 0.685**

El AUC baja drásticamente respecto al modelo binario (0.991 → 0.685). Esto confirma lo que anticipábamos: el resultado del modelo binario estaba inflado por el leakage de `n_meses_facturados`. Sin esa variable trampa, el AUC real del problema es 0.685. Es un resultado honesto y razonable para un problema de churn mensual.

**Interpretación de los coeficientes — modelo temporal:**

Variables que más aumentan el riesgo de churn (barras rojas):
- `calidad_global_lag1` → es el predictor más potente. Si la calidad de red del mes anterior era mala, el riesgo de abandono ese mes sube considerablemente.
- `impago_mes_lag1` e `impago_mes_roll3` → un impago el mes anterior o una tendencia de impagos en los últimos 3 meses son señales de alarma claras.
- `estado_civil_Divorciado/a` → segmento con mayor propensión al abandono.
- `tipo_dispositivo_Gama baja` → clientes con dispositivos de gama baja churneán más.
- `stress_mes_roll3` → el estrés de red acumulado en los últimos 3 meses sube el riesgo.

Variables que más reducen el riesgo (barras azules):
- `sexo_F` y `sexo_M` → respecto a la categoría de referencia (desconocido), tener sexo registrado reduce el riesgo. Puede indicar que los clientes con datos más completos son más comprometidos.
- `tipo_zona_urbana_premium` → vivir en zona premium reduce el riesgo, confirmando lo que vimos en el EDA.
- `antiguedad_meses` → a mayor antigüedad, menos riesgo. Coincide con el EDA.
- `cobertura_5g_lag1` → mejor cobertura 5G el mes anterior reduce el riesgo.

⚠️ Nota importante sobre `calidad_global_lag1`: aparece como el predictor más fuerte aumentando el churn, lo cual es contraintuitivo (más calidad = más churn). Esto puede deberse a multicolinealidad con otras variables de red o a que el índice de calidad global no mide lo que esperamos en este contexto. Vale la pena investigarlo más.


## 8. Modelo 2 — Random Forest


In [ ]:
pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

pipe_rf.fit(X_train, y_train)
y_pred_proba_rf = pipe_rf.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f'Random Forest — AUC en test: {auc_rf:.3f}')

# Importancia de variables
nom_feature_names_rf = (pipe_rf.named_steps['preprocessor']
                        .named_transformers_['nom']
                        .named_steps['encoder']
                        .get_feature_names_out(FEATURES_CAT_NOMINAL))
feature_names_rf = FEATURES_NUM + list(nom_feature_names_rf) + FEATURES_ORDINAL

importancias = pd.Series(
    pipe_rf.named_steps['model'].feature_importances_,
    index=feature_names_rf
).sort_values(ascending=False)

top15 = importancias.head(15).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
top15.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Random Forest Temporal — Top 15 variables más importantes',
             fontweight='bold')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

print('\nTop 10 variables:')
print(importancias.head(10).round(4).to_string())

### Resultado — Random Forest Temporal

**AUC en test: 0.668**

El Random Forest obtiene un AUC ligeramente inferior a la Regresión Logística (0.668 vs 0.685) en el modelo temporal. Esto es interesante porque en el modelo binario era al revés. En problemas con mucho ruido y datos panel, los modelos lineales a veces generalizan mejor que los árboles.

**Importancia de variables — Random Forest Temporal:**

A diferencia del modelo binario donde `n_meses_facturados` dominaba con el 45%, aquí las variables están mucho más repartidas, lo que es una señal de que el modelo está aprendiendo relaciones más diversas:

- `dias_retraso_mes_lag1` (8.5%) → el número de días de retraso en el pago del mes anterior es el predictor más importante. Un cliente que retrasó el pago el mes pasado tiene más riesgo de irse.
- `ingreso_estimado` (7.3%) → los ingresos del cliente influyen, confirmando el EDA donde veíamos que clientes con menos ingresos abandonan más.
- `importe_mes_lag1` (6.9%) → el importe facturado el mes anterior.
- `stress_mes_lag1` (5.9%) y `calidad_global_lag1` (5.8%) → la calidad de red del mes anterior tiene peso consistente en ambos modelos.
- `antiguedad_meses` (5.8%) → confirmado: la antigüedad protege contra el churn.
- Las variables de rolling window (`variacion_consumo_roll3`, `stress_mes_roll3`) también aparecen, confirmando que las tendencias recientes importan más que un solo mes.


---
## 9. Comparación completa de modelos

Comparamos los cuatro modelos del proyecto para contar la historia completa.


In [ ]:
# AUCs del modelo binario (obtenidos en el notebook anterior)
AUC_BINARIO_LR = 0.991
AUC_BINARIO_RF = 0.994

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curvas ROC del modelo temporal
RocCurveDisplay.from_predictions(
    y_test, y_pred_proba_lr,
    name=f'Temporal — Reg. Logística (AUC={auc_lr:.3f})',
    color='steelblue', ax=axes[0]
)
RocCurveDisplay.from_predictions(
    y_test, y_pred_proba_rf,
    name=f'Temporal — Random Forest (AUC={auc_rf:.3f})',
    color='#E85C4C', ax=axes[0]
)
axes[0].plot([0,1],[0,1], 'k--', label='Baseline (AUC=0.500)')
axes[0].set_title('Curva ROC — Modelos temporales', fontweight='bold')
axes[0].legend(fontsize=8)

# Comparativa de los 4 modelos
modelos = ['Binario\nReg. Logística', 'Binario\nRandom Forest',
           'Temporal\nReg. Logística', 'Temporal\nRandom Forest']
aucs    = [AUC_BINARIO_LR, AUC_BINARIO_RF, auc_lr, auc_rf]
colores = ['#a8c6e8', '#4C9BE8', '#f5a4a0', '#E85C4C']

bars = axes[1].bar(modelos, aucs, color=colores, edgecolor='white')
axes[1].set_ylim(0.5, 1.02)
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('Comparativa AUC — Los 4 modelos del proyecto', fontweight='bold')
axes[1].set_ylabel('AUC-ROC')
for bar, val in zip(bars, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n=== RESUMEN COMPARATIVO ===')
resumen = pd.DataFrame({
    'Modelo':        modelos,
    'AUC':           aucs,
    'Leakage':       ['Sí', 'Sí', 'No', 'No'],
    'Uso producción':['No', 'No', 'Sí', 'Sí'],
})
display(resumen)

### Resultado — Comparación de los 4 modelos

| Modelo | AUC | Leakage | Válido para producción |
|--------|-----|---------|------------------------|
| Binario — Reg. Logística | 0.991 | Sí | No |
| Binario — Random Forest | 0.994 | Sí | No |
| Temporal — Reg. Logística | **0.685** | No | **Sí** |
| Temporal — Random Forest | 0.668 | No | Sí |

**¿Por qué baja tanto el AUC del modelo temporal?**

La caída de ~0.31 puntos de AUC (de 0.994 a 0.668) cuantifica exactamente cuánto del resultado del modelo binario era artificial. La variable `n_meses_facturados` actuaba como proxy perfecto del target: los clientes con pocos meses eran los que churneaban, y los modelos lo detectaban trivialmente. Al eliminar esa trampa y obligar al modelo a predecir con solo información del pasado, el AUC refleja la dificultad real del problema.

Un AUC de 0.685 en un problema de churn mensual con solo 0.64% de eventos positivos es un resultado **razonable y útil** para el negocio. No es perfecto, pero permite identificar clientes en riesgo con suficiente antelación.

**¿Por qué la Regresión Logística supera al Random Forest en el modelo temporal?**

En el modelo binario el RF ganaba porque el leakage creaba relaciones no lineales fáciles de capturar con árboles. En el modelo temporal, sin ese atajo, el problema es más difícil y las relaciones son más lineales. La Regresión Logística generaliza mejor en este escenario.

**Modelo elegido para producción: Regresión Logística Temporal (AUC = 0.685)**

- Mejor AUC de los modelos sin leakage
- Totalmente interpretable: los coeficientes explican directamente qué variables suben o bajan el riesgo
- Genera probabilidades mensuales por cliente, listas para una campaña de retención


---
## 10. Aplicación práctica: clientes en riesgo

El modelo temporal produce cada mes una lista de clientes con su probabilidad de churn. Simulamos cómo se usaría en producción analizando todos los meses del conjunto de test.

**Sobre el Top 10**: muestra los 10 registros con mayor probabilidad de todo el histórico de test (62.524 filas). Al ser los máximos absolutos, todos superan el umbral alto (0.05) y se clasifican como 'Alto'. Son la cola derecha del histograma.

**Sobre la distribución general**: la gran mayoría de registros tiene probabilidad baja porque la tasa de churn mensual es solo del 0.63%. En la mayoría de meses, la mayoría de clientes no churnearon. Por eso el grueso cae en 'Bajo'.

**Lo más importante es la tabla de tasas reales por nivel de riesgo**, que confirma que el modelo discrimina correctamente:
- Clientes marcados como **Bajo**: tasa de churn real del 0.6%
- Clientes marcados como **Medio**: tasa de churn real del 2.3%
- Clientes marcados como **Alto**: tasa de churn real del 2.7%

Los clientes clasificados como 'Alto' churnearon **4 veces más** que los de 'Bajo'. No es perfecto, pero es accionable para el negocio: concentrar las campañas de retención en los 182 registros de riesgo alto en lugar de contactar a los 62.524 totales.


In [ ]:
# Probabilidades sobre todo el conjunto de test (más representativo)
test_full = test.copy()
test_full['prob_churn'] = pipe_lr.predict_proba(X_test)[:, 1]
test_full['riesgo'] = pd.cut(
    test_full['prob_churn'],
    bins=[0, 0.02, 0.05, 1.0],
    labels=['Bajo', 'Medio', 'Alto']
)

print(f'Clientes-mes en test: {len(test_full):,}')
print(f'Tasa de churn real:   {test_full["churn"].mean()*100:.2f}%')
print()
print('Distribución por nivel de riesgo:')
print(test_full['riesgo'].value_counts().sort_index())

# Top 10 con mayor probabilidad
print('\nTop 10 clientes-mes con mayor probabilidad de churn:')
cols_show = ['cliente_id', 'mes', 'prob_churn', 'riesgo', 'churn']
cols_disp = [c for c in cols_show if c in test_full.columns]
display(test_full.nlargest(10, 'prob_churn')[cols_disp].reset_index(drop=True))

# Precisión por nivel de riesgo: ¿qué % de los que marcamos como Alto realmente churnearon?
print('\nTasa de churn real por nivel de riesgo asignado:')
print(test_full.groupby('riesgo', observed=True)['churn'].agg(['mean','count'])
      .rename(columns={'mean':'tasa_churn_real','count':'n_registros'})
      .round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribución de probabilidades
test_full['prob_churn'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(0.02, color='orange', linestyle='--', label='Umbral medio (0.02)')
axes[0].axvline(0.05, color='red', linestyle='--', label='Umbral alto (0.05)')
axes[0].set_title('Distribución de probabilidades de churn\n(todos los meses en test)',
                  fontweight='bold')
axes[0].set_xlabel('Probabilidad de churn')
axes[0].legend(fontsize=8)

# Clientes por nivel de riesgo
riesgo_counts = test_full['riesgo'].value_counts().sort_index()
colors_riesgo = ['#4C9BE8', '#f39c12', '#E85C4C']
axes[1].bar(riesgo_counts.index, riesgo_counts.values, color=colors_riesgo)
axes[1].set_title('Registros por nivel de riesgo\n(todos los meses en test)', fontweight='bold')
axes[1].set_ylabel('Número de registros cliente-mes')
for bar, val in zip(axes[1].patches, riesgo_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

---
## 11. Conclusiones

### Comparativa final de los 4 modelos

| Modelo | AUC | Leakage | Válido para producción |
|--------|-----|---------|------------------------|
| Binario — Reg. Logística | 0.991 | Sí | No |
| Binario — Random Forest | 0.994 | Sí | No |
| **Temporal — Reg. Logística** | **0.685** | **No** | **Sí** |
| Temporal — Random Forest | 0.668 | No | Sí |

### ¿Por qué el AUC temporal es menor?

La caída de 0.994 a 0.668 cuantifica el impacto del leakage en el modelo binario. La variable `n_meses_facturados` era un proxy del target: los churners tienen pocos meses porque se fueron antes. Al eliminarla y usar solo información del pasado, el AUC refleja la dificultad real del problema.

Un AUC de 0.685 con una tasa de churn mensual del 0.64% es un resultado válido y útil para el negocio.

### Variables más importantes del modelo final (Temporal — Reg. Logística)

Aumentan el riesgo de churn:
1. `calidad_global_lag1` — mala calidad de red el mes anterior
2. `impago_mes_lag1` — impago en el mes anterior
3. `impago_mes_roll3` — tendencia de impagos en los últimos 3 meses
4. `stress_mes_roll3` — estrés de red acumulado en los últimos 3 meses

Reducen el riesgo de churn:
1. `tipo_zona_urbana_premium` — vivir en zona urbana premium protege
2. `antiguedad_meses` — más tiempo como cliente, menos riesgo
3. `cobertura_5g_lag1` — mejor cobertura 5G el mes anterior
4. `ingreso_estimado` — mayores ingresos se asocian a menos churn

### Modelo recomendado para producción

**Regresión Logística Temporal** por tres razones:
- Mejor AUC entre los modelos sin leakage (0.685)
- Totalmente interpretable — se puede explicar a negocio qué cliente está en riesgo y por qué
- Genera cada mes una lista de clientes con su probabilidad de churn, lista para ser consumida por el equipo comercial

### Uso operativo

Cada mes, el modelo genera una lista priorizada:
- **Riesgo alto** (prob > 0.05): contacto prioritario con oferta de retención
- **Riesgo medio** (0.02 - 0.05): seguimiento y comunicación proactiva
- **Riesgo bajo** (prob < 0.02): no actuar, evitar costes innecesarios

---
*Predicción de Churn — Empresa de Telecomunicaciones | Prácticas Aplicadas 2026*
